# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hamzaabutt/flyrank/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This notebook turns our validated Random Forest model probabilities into an actionable content refresh playbook with reason codes, human review guards, and retrain triggers.

## 1. Ranked actions + reason codes

**Action Playbook Strategy Matrix:**
- `expand_and_refresh`: Triggered for thin visible pages (`word_count < 1200`). Action: Comprehensive content expansion, adding key subtopics and structured FAQ sections.
- `refresh_and_review_ctr`: Triggered for low CTR visible pages (`ctr < 0.5%`). Action: Title tag overhaul, meta description optimization, and Rich Snippet schema addition.
- `refresh`: Triggered for stale or decaying pages (`days_since_last_update >= 180` or `trend_direction == 'down'`). Action: Update stats, replace dead links, refresh publish timestamp.
- `monitor`: Low risk pages maintained on standard monitoring cycles.

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
import numpy as np

# Set path relative to repo root
repo_root = Path("../../").resolve() if Path("../../scripts").exists() else Path("../").resolve()
sys.path.append(str(repo_root / "scripts"))
from ml_utils import PROCESSED_DIR, OUTPUT_DIR

# Load generated refresh queue
queue_path = OUTPUT_DIR / "refresh_queue.csv"
if not queue_path.exists():
    queue_path = repo_root / "outputs" / "refresh_queue.csv"

df_queue = pd.read_csv(queue_path)
print(f"Loaded Final Refresh Queue: {len(df_queue):,} ranked URLs.")
df_queue[['final_rank', 'content_id', 'final_refresh_score', 'suggested_action', 'reason_codes']].head(10)

## 2. Intended use and limits

**Intended Use:**
- Serves content strategists and SEO editorial leads to prioritize weekly content update sprints.
- Replaces subjective guessing with empirical data-driven scoring.

**Operational Limits:**
- Does not auto-generate copy; human writers perform all content updates.
- Cannot predict sudden real-world news trends or unannounced search engine algorithm updates.

In [ ]:
action_summary = df_queue['suggested_action'].value_counts().rename('URL Count').to_frame()
action_summary['Percentage'] = (action_summary['URL Count'] / len(df_queue) * 100).map('{:.1f}%'.format)
print("Action Playbook Queue Distribution:")
action_summary

## 3. Human review + the no-go list

**Human Review Protocol:**
1. **SERP Intent Check**: Verify that target keyword search intent hasn't shifted from informational to transactional.
2. **Brand & Voice Verification**: Ensure added sections align with corporate messaging guidelines.

**The No-Go List (Never Automate):**
- Never prune or redirect high-converting URLs, regardless of word count.
- Never apply automated content rewrites to YMYL (Your Money Your Life) or legal/medical topics without expert approval.

In [ ]:
top_actions = df_queue.head(20)[['final_rank', 'content_id', 'suggested_action', 'reason_codes', 'impressions_90d']]
print("Top 20 Priority Review Candidates:")
top_actions

## 4. Monitoring / retrain triggers

**System Monitoring & Retrain Triggers:**
- **Monthly Retrain Schedule**: Re-fit models every 30 days as new monthly Search Console data ingests.
- **Performance Degradation Trigger**: If test set Precision@50 falls below 60.0% during monthly validation, trigger feature re-engineering.
- **Algorithm Shift Trigger**: Re-evaluate baseline when major Search Engine core updates occur.

In [ ]:
print("Monitoring & Retrain Rules Initialized.")
print("- Monthly Retrain Window: 30 days")
print("- Precision Threshold: >= 60.0% P@50")

## 5. Exports for the paper

We export the final playbook queue to `work/outputs/final_action_playbook.csv` for reproducibility and paper inclusion.

In [ ]:
output_playbook = repo_root / "work" / "outputs" / "final_action_playbook.csv"
output_playbook.parent.mkdir(parents=True, exist_ok=True)
df_queue.to_csv(output_playbook, index=False)
print(f"Exported final playbook to {output_playbook}")

## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/`